In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
!import sys
!{sys.executable} -m pip install transformers

/bin/bash: line 1: import: command not found
/bin/bash: line 1: {sys.executable}: command not found


In [4]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio

In [8]:
import torch

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline

**Initialize Model, Tokenizer**

In [10]:
model = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModelForCausalLM.from_pretrained(model)

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 343.58it/s]


**Using Hugging face transformer pipeline for text generation**

In [53]:
gen_pipeline = pipeline(task="text-generation", model = model, tokenizer = tokenizer)

In [54]:
gen_pipeline("Hello how are you", max_new_tokens=25)

Both `max_new_tokens` (=25) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "Hello how are you?\n\nI'm an AI assistant, so I don't have feelings like humans do. However, I'm here to help you"}]

**Use Hugging face transformers functions for text generation**

In [19]:
prompts = ["what is the current temperature", "What is the capital of India"]

In [22]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
tokenized = tokenizer(prompts, return_tensors="pt", padding=True)

In [25]:
tokenized['input_ids'].shape

torch.Size([2, 6])

In [26]:
tokenizer.batch_decode(tokenized['input_ids'])

['<|endoftext|>what is the current temperature',
 'What is the capital of India']

In [26]:
chat_template = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<|im_start|>assistant\n' }}"
    "{% endif %}"
)

tokenizer.chat_template = chat_template

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [62]:
prompts = [
    {"role": "system", "content": "You are a advisor for online banking."},
    {"role": "user", "content": "Give details on best way to change online banking password."}
]
tokenized = tokenizer.apply_chat_template(
    prompts,
    add_generation_prompt=True,
    padding=True,
    tokenize=True,
    return_tensors="pt"
)

In [57]:
input_tokens = tokenized['input_ids']
tokenizer.batch_decode(input_tokens)

['<|im_start|>system\nYou are a advisor for online banking.<|im_end|>\n<|im_start|>user\nGive details on best way to change online banking password.<|im_end|>\n<|im_start|>assistant\n']

In [63]:
generated_ids = model.generate(**tokenized, max_new_tokens=50,pad_token_id=tokenizer.eos_token_id,do_sample=True,temperature=0.7)

In [59]:
tokenizer.batch_decode(generated_ids)

['<|im_start|>system\nYou are a advisor for online banking.<|im_end|>\n<|im_start|>user\nGive details on best way to change online banking password.<|im_end|>\n<|im_start|>assistant\nTo change your online banking password, follow these detailed steps:\n\n1. **Log in**: First, log into your account using the secure login page provided by your bank.\n\n2. **Navigate to Account Settings or Password Management**: Once logged in, look']

In [7]:
text = "hello how are"
tokenized = tokenizer([text], return_tensors="pt")
out = model(**tokenized)

In [18]:
lasttoken = out.logits[0,-1].argmax(axis=-1)

In [19]:
tokenizer.decode(lasttoken)

' you'

**Before Finetuning, testing what model emits**

In [20]:
prompts = [
    {"role": "user", "content": "Who to subscribe in Youtube for Machine Learning."},
    {"role": "assitant", "content": "Subscribe to "}
]

In [24]:
tokenized = tokenizer.apply_chat_template(prompts, padding=True, tokenize=True,continue_final_message=True,return_tensors="pt")

In [27]:
tokenized["input_ids"].shape

torch.Size([1, 21])

In [16]:
output = model.generate(**tokenized, temperature=0.5, max_new_tokens=50)

In [18]:
tokenizer.batch_decode(output)

['<|im_start|>user\nWho to subscribe in Youtube for Machine Learning.<|im_end|>\n<|im_start|>assitant\nSubscribe to. 1) YouTube channel of Google AI - This channel contains a wide range of videos on various topics related to machine learning, including deep learning, natural language processing, computer vision, and more.\n\n2) YouTube channel of Andrew Ng - He is one']

In [19]:
target_response = "Learning with ML"

**Supervised FineTuning - Updating model weights**

In [24]:
def generate_input_output_pair(prompt, target_responses):
    # 1. Apply chat template to format the prompt correctly
    chat_template = tokenizer.apply_chat_template(prompt, tokenize=False, continue_final_message=True)
    
    # 2. Append the target response + EOS token
    full_response_text = [
        (chat + " " + response + tokenizer.eos_token) # Note: .eos_token string, not .add_eos_token
        for chat, response in zip(chat_template, target_responses)
    ]

    # 3. Tokenize the full sequence
    input_ids_tokenized = tokenizer(full_response_text, return_tensors="pt", padding=True, add_special_tokens=False)["input_ids"]

    # 4. Create labels (masking out prompt tokens with -100)
    # We want labels to have the same shape as input_ids
    # labels_tokenized = tokenizer(
    #    [" " + response + tokenizer.eos_token for response in target_responses], 
    #    add_special_tokens=False, 
    #    return_tensors="pt", 
    #    padding="max_length", 
    #    max_length=input_ids_tokenized.shape[1]
    #)["input_ids"]

    labels_tokenized = input_ids_tokenized.clone()

    print(tokenizer.batch_decode(input_ids_tokenized))
    print(tokenizer.batch_decode(labels_tokenized))

    # 5. Math: Mask padding and prompt area
    # If it's a pad token, set to -100 so the loss function ignores it
    labels_tokenized_fixed = torch.where(labels_tokenized != tokenizer.pad_token_id, labels_tokenized, -100)
    print(labels_tokenized)
    print(labels_tokenized_fixed)

    # 6. Shifting for Next-Token Prediction
    # The input is the sequence minus the last token
    input_ids_final = input_ids_tokenized[:, :-1]
    # The labels are the sequence shifted right (target is the 'next' token)
    labels_final = labels_tokenized_fixed[:, 1:]

    print(input_ids_final)
    print(labels_final)

    # 7. Attention Mask (Must match the shifted input size)
    attention_mask = (input_ids_final != tokenizer.pad_token_id).long()
    print(attention_mask)

    print(len(attention_mask[0]), len(labels_final[0]))

    return {
        "input_ids": input_ids_final,
        "labels": labels_final,
        "attention_mask": attention_mask
    }

In [21]:
import torch.nn as nn
def calculate_loss(logits, labels):
    loss_fn = nn.CrossEntropyLoss(reduction="none")
    return loss_fn(logits.view(-1, logits.size(-1)) , labels.view(-1))

In [22]:
from torch.optim import AdamW

In [27]:
data = generate_input_output_pair([prompts], [target_response])

['<|im_start|>user\nWho to subscribe in Youtube for Machine Learning.<|im_end|>\n<|im_start|>assitant\nSubscribe to  Learning with ML<|im_end|>']
['<|im_start|>user\nWho to subscribe in Youtube for Machine Learning.<|im_end|>\n<|im_start|>assitant\nSubscribe to  Learning with ML<|im_end|>']
tensor([[151644,    872,    198,  15191,    311,  17963,    304,  37303,    369,
          12960,  20909,     13, 151645,    198, 151644,    395,  50944,    198,
          28573,    311,    220,  20909,    448,  19614, 151645]])
tensor([[151644,    872,    198,  15191,    311,  17963,    304,  37303,    369,
          12960,  20909,     13, 151645,    198, 151644,    395,  50944,    198,
          28573,    311,    220,  20909,    448,  19614, 151645]])
tensor([[151644,    872,    198,  15191,    311,  17963,    304,  37303,    369,
          12960,  20909,     13, 151645,    198, 151644,    395,  50944,    198,
          28573,    311,    220,  20909,    448,  19614]])
tensor([[   872,    198,  151

In [15]:
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

In [16]:
for _ in range(5):
    out = model(input_ids=data["input_ids"])
    loss = calculate_loss(out.logits, data["labels"]).mean()
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    print(loss.item())

7.34375
6.03125
3.015625
1.5546875
0.404296875


**After finetuning, what model emits**

In [17]:
tokenized = tokenizer.apply_chat_template(prompts, padding=True, return_tensors="pt", tokenize=True, continue_final_message=True)

In [19]:
output = model.generate(**tokenized, temperature=0.7, max_new_tokens=50)

In [20]:
tokenizer.batch_decode(output)

['<|im_start|>user\nWho to subscribe in Youtube for Machine Learning.<|im_end|>\n<|im_start|>assitant\nSubscribe to  Learning with ML<|im_end|>']

**LORA Finetuning**

In [3]:
import sys
!{sys.executable} -m pip install peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 16.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [peft]1/2 [peft]


In [4]:
from peft import LoraConfig, get_peft_model

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [16]:
lora_config = LoraConfig(task_type="CAUSAL_LM", r=16,lora_alpha=16,lora_dropout=0.1,target_modules=['q_proj', 'v_proj'])

In [17]:
lora_model = get_peft_model(model, lora_config)

In [18]:
lora_model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [28]:
optimizer = AdamW(lora_model.parameters(), lr=1e-4, weight_decay=0.01)

In [34]:
for _ in range(50):
    out = lora_model(input_ids=data["input_ids"])
    loss = calculate_loss(out.logits, data["labels"]).mean()
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    print(loss.item())

3.171875
3.078125
3.0
2.921875
2.828125
2.734375
2.640625
2.5625
2.46875
2.375
2.265625
2.15625
2.015625
1.859375
1.7421875
1.640625
1.5078125
1.3671875
1.25
1.1875
1.1171875
1.046875
1.0
0.9609375
0.90625
0.87890625
0.86328125
0.8125
0.7890625
0.74609375
0.71875
0.69921875
0.6796875
0.6484375
0.6171875
0.6015625
0.5546875
0.51953125
0.4765625
0.46484375
0.4609375
0.453125
0.439453125
0.43359375
0.443359375
0.42578125
0.421875
0.4140625
0.412109375
0.404296875


In [35]:
tokenized = tokenizer.apply_chat_template(prompts, padding=True, return_tensors="pt", tokenize=True, continue_final_message=True)
output = lora_model.generate(**tokenized, temperature=0.7, max_new_tokens=50)


In [36]:
tokenizer.batch_decode(output)

['<|im_start|>user\nWho to subscribe in Youtube for Machine Learning.<|im_end|>\n<|im_start|>assitant\nSubscribe to  Learning with ML<|im_end|>']